
# 作業一：Prompt Engineering 實作

> 請「先」在檔名中填入你的學號再上傳（在下方 **[提交與輸出]** 有自動命名）。  
> 完成後請依課程規定提交。

---

## 規劃與來源
本 Notebook 依據課程投影片之三個任務設計：
- 任務一：Email 格式轉換（角色設定、格式控制、語氣調整）  
- 任務二：文章摘要（使用 Chain-of-Thought）  
- 任務三：翻譯與語氣調整（多重指令整合、文化語境）  

請根據各章節中的 **TODO** 完成核心實作。助教已替你準備好輸入範例、評分提示、與基本工具函式。



## 注意事項與遲交規則（節錄）
- 遲交扣分：每延遲一天扣 **10%**，最多寬限 **3** 天；超過不受理，記 **0** 分。  
- 以繳交系統時間為準。  
- 請保留你的設計思路（尤其是任務二的 CoT 步驟）。

> 提醒：好的 Prompt = 清楚、具體、有條理。



## 0. 環境與套件
> 本模板預留了與 LLM 互動的介面，但**不綁定特定供應商**。你可以選擇 OpenAI、Anthropic、Azure OpenAI、Google 等。  
> 若你使用 Colab，請先安裝對應的 Python SDK（例如 `openai`）。


In [ ]:
from google.colab import userdata
userdata.get('OpenapiKey')

In [ ]:
# === 可選：安裝你要用的 SDK（Colab 環境）===
# 例如：
!pip -q install openai

import os
from dataclasses import dataclass
from typing import Optional, Dict, Any

# ========== 可重用：統一的 LLM 呼叫介面 ==========
@dataclass
class LLMConfig:
    provider: str = "openai"
    model: str = "gpt-3.5-turbo"     # TODO: 指定模型名稱（必填）
    api_key_env: str = "OPENAI_API_KEY"     # TODO: 指定 API Key 環境變數名

def call_llm(prompt: str, system: Optional[str]=None, cfg: Optional[LLMConfig]=None, **kwargs) -> str:
    """
    統一的 LLM 呼叫函式（簡化版）。
    - 你可以在此實作實際的 SDK 呼叫；或在各任務中自行呼叫不同供應商的 API。
    - 若不想用雲端 API，可改為你自定義的規則程式碼（但需在報告中說明限制）。
    """
    cfg = cfg or LLMConfig()
    provider = cfg.provider.lower()
    os.environ["OPENAI_API_KEY"] = userdata.get('OpenapiKey')  # TODO: 對應的 API Key 環境變數名

    # TODO: 依你選定的 provider 完成實作（下方示例為 OpenAI 的「佔位」程式碼片段）
    # ===== OpenAI （需: pip install openai）=====
    if provider == "openai":
        from openai import OpenAI
        client = OpenAI(api_key=os.getenv(cfg.api_key_env))
        messages = []
        if system:
            messages.append({ "role": "system", "content": system })
        messages.append({ "role": "user", "content": prompt })
        resp = client.chat.completions.create(model=cfg.model, messages=messages, **kwargs)
        return resp.choices[0].message.content
        raise NotImplementedError("TODO: 請在 call_llm() 中實作 OpenAI呼叫。")
    else:
        raise NotImplementedError(f"TODO: 尚未支援 provider={provider!r}。")


In [ ]:

# ========== 共用工具 ==========
def count_zh_chars(text: str) -> int:
    """簡易字數統計（中文為主，含中英文混排時可視需要自行調整）。"""
    return len(text.strip())

def save_text(path: str, content: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)

def load_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()



---
# 任務一：Email 格式轉換

**學習重點**：角色設定、格式控制、語氣調整  
**輸入範例**：
> 嘿老闆，明天10點的會議我可能會遲到一下，大概15分鐘吧，昨晚小孩發燒，早上得先送他去看醫生。會議的資料我都準備好了，可以先請小王開始報告嗎？

**任務要求**：
- 將上述訊息改寫成**正式商業郵件**
- 必須包含：**主旨、稱謂、清楚說明、解決方案、專業結尾**
- **保持原意**但提升專業度
- 在 Prompt 中**展示你的設計思路**（例如列出需求、格式、語氣等）

**評分重點**：Prompt 清晰度與結構性、格式控制是否成功、語氣轉換自然度


In [ ]:

# ===== TODO(學生)：設計你的 Prompt（可含 system + user）原始資訊=====
email_raw = (
    "嘿老闆，明天10點的會議我可能會遲到一下，大概15分鐘吧，"
    "昨晚小孩發燒，早上得先送他去看醫生。會議的資料我都準備好了，"
    "可以先請小王開始報告嗎？"
)
# ====== 輸入prompt ===========
system_email = """
角色：你是資深商務寫手。
背景：將非正式訊息改寫成寄給上級或會議主持人的正式郵件。
對象：公司主管或老師。
必備：主旨、稱謂、清楚說明、解決方案、專業結尾。
限制：保留原意、不虛構事實、不透露敏感個資、避免口語和表情符號。
輸出：第一行「主旨: ……」，接著為段落化郵件內容。
請輸出正式郵件，在輸出前確認是否有達成每項任務需求：
郵件語氣專業、精煉。不輸出任何解說。
輸出的信件內容念起來要是順暢的
如果沒有額外說明誰寫的 以"林羽航"作為寫信的人
不需要額外解釋
""".strip()

# === 在此撰寫你的 user prompt，需明確描述格式要求與轉換目標 +raw的prompt ===
user_email_prompt = f"""
【任務】
給我把下方【原始訊息】改寫成正式商業郵件，並確保格式與語氣符合 system 的要求。
【任務要求】
- 第一行一定要「主旨: ...」 並符合內容重點
- 內容分段清楚（段落之間空行）
- 不要添加過多額外內容
- 保留原意，但提升禮貌與專業
【原始訊息】
{email_raw}
"""

# === 呼叫 LLM（或你自定義的規則程式）===
# TODO: 取消下一行註解，並完成 call_llm 的實作
email_formal = call_llm(user_email_prompt, system=system_email)

# 暫時：請同學自行將結果貼到此變數（若尚未串接 API），以便後續存檔。已經串接好openai的可以刪掉這行
#email_formal = """
#TODO：在這裡貼上你產生的正式商業郵件結果。
#""".strip()

# 儲存輸出
save_text("outputs/task1_email_formal.txt", email_formal)
print("任務一完成（暫存）。輸出檔：outputs/task1_email_formal.txt")

任務一完成（暫存）。輸出檔：outputs/task1_email_formal.txt



---
# 任務二：文章摘要（使用 CoT）

**輸入文章（約 200 字）**：
> 根據最新研究顯示，台灣的科技產業正面臨人才短缺的嚴峻挑戰。調查發現，有超過70%的科技公司表示難以找到合適的AI和資料科學人才。這個問題的根源包括：教育體系與產業需求脫節、薪資競爭力不足，以及人才外流至海外市場。為了解決這個問題，政府推出了多項措施，包括增加大學相關科系名額、提供企業培訓補助，以及放寬外籍人才引進政策。然而，專家認為這些措施需要時間才能看到成效，短期內企業仍需要自行投資人才培育。

**任務要求**：
- 使用 **Chain-of-Thought** 方法設計 Prompt
- **明確展示分析步驟**（例：主題識別 → 要點提取 → 重要性排序 → 總結）
- 產生 **50 字以內** 的精簡摘要（需保留關鍵資訊）

**評分重點**：
- CoT 步驟的邏輯性與完整性
- 摘要的精確度與資訊保留度
- 是否展現結構化思考過程


In [ ]:

article = (
    """根據最新研究顯示，台灣的科技產業正面臨人才短缺的嚴峻挑戰。調查發現，有超過70%的科技公司表示難以找到合適的AI和資料科學人才。這個問題的根源包括：教育體系與產業需求脫節、薪資競爭力不足，以及人才外流至海外市場。為了解決這個問題，政府推出了多項措施，包括增加大學相關科系名額、提供企業培訓補助，以及放寬外籍人才引進政策。然而，專家認為這些措施需要時間才能看到成效，短期內企業仍需要自行投資人才培育。"""
)

# ===== TODO(學生)：設計你的 CoT Prompt =====
system_cot = """
你是一位專業的資訊分析師與文本摘要專家 給我嚴格按照使用者要求。

你擅長使用「Chain-of-Thought」方法，透過一步一步的邏輯推導來解構複雜的資訊。

在執行任務時，你會嚴格按照使用者要求的思考步驟來展示你的完整分析過程，然後才基於該分析，產出精確且符合字數限制的最終摘要。

格式類似：
[分析步驟]
(1) 主題識別：...
(2) 要點提取：...
(3) 重要性排序：...
[最終摘要]
""".strip()

user_cot_prompt = f"""
仔細閱讀以下文章後，先理解文章內容以及使用者要求
接著依指示列出分析步驟
明確展示分析步驟 越清楚越好
至少需要 主題識別 → 要點提取 → 重要性排序 → 總結

請依照以下輸出：
Chain-of-Thought 思考過程
最後列出 [最終摘要] 需保留關鍵資訊
[最終摘要] 需要絕對保證字數在50字內 否則會有嚴重錯誤

!! 輸出前確認[最終摘要]的字數 < 50 字 在輸出
確保輸出格式段落間空一行
[文章]
{article}
"""

# TODO: 使用 LLM（或你自定義的規則程式）產生輸出
cot_output = call_llm(user_cot_prompt, system=system_cot)



# 50字內檢查（僅對「最終摘要」行做粗略偵測）
lines = [ln for ln in cot_output.splitlines() if ln.strip()]  # 移除空白行
final_lines = []

# 掃描每一行，找到 [最終摘要]，並抓取下一行作為摘要內容
for i, ln in enumerate(lines):
    if ln.strip().startswith("[最終摘要]") or ln.strip().startswith("最終摘要"):
        # 若下一行存在且不是空白，就取下一行
        if i + 1 < len(lines) and lines[i + 1].strip():
            final_lines.append(lines[i + 1].strip())
if final_lines:
    final = final_lines[-1]  # 取最後一個摘要內容
    n_chars = count_zh_chars(final)
    print(f"最終摘要字數(有加標點符號)：約 {n_chars} 字（目標 <= 50）")
else:
    print("⚠️ 未偵測到最終摘要段落，請檢查輸出格式。")

save_text("outputs/task2_summary_cot.txt", cot_output)
print("任務二完成（暫存）。輸出檔：outputs/task2_summary_cot.txt")

最終摘要字數(有加標點符號)：約 64 字（目標 <= 50）
任務二完成（暫存）。輸出檔：outputs/task2_summary_cot.txt



---
# 任務三：翻譯與語氣調整（多重指令）

**輸入句子**：  
> I'm afraid I cannot attend tomorrow's meeting due to a scheduling conflict. Would it be possible to reschedule to next week?

**任務要求**：
- 翻譯成**繁體中文**
- 調整為**友善、輕鬆**的語氣（像和好朋友說話）
- 保持原意並加入台灣常用語氣
- **設計一個 Prompt** 能同時完成以上所有要求  
**進階加分**：
- 讓 AI 提供 **2–3 種不同語氣程度** 的版本，並**說明各版本適用情境**。


In [ ]:

en_sentence = "I'm afraid I cannot attend tomorrow's meeting due to a scheduling conflict. Would it be possible to reschedule to next week?"

# ===== TODO(學生)：設計能同時完成「翻譯＋語氣調整」的 Prompt =====
system_trans = """
你是一位熟悉台灣語境的中英翻譯與語氣調整專家。

你的任務是：
1. 將英文句子翻譯為 "自然的繁體中文"
2. 語氣要友善、輕鬆，但仍保留禮貌與清楚的意思。
3. 加入台灣常用語氣
4. 若要求多版本，請依不同語氣程度提供翻譯，並說明各版本適用情境。
""".strip()

user_trans_prompt = f"""
將下列英文句子轉為繁體中文，語氣友善、輕鬆，並保留原意：
{en_sentence}
【風格範例】
英文：I'm sorry I'll be late.
自然口語：不好意思，我會晚一點到。
中性禮貌：抱歉，我可能會稍微晚到。
正式：很抱歉，我恐怕將延後幾分鐘抵達。
請提供三個版本：

【版本1｜自然口語版】語氣像台灣朋友聊天或對話中會用的自然說法

【版本2｜中性禮貌版】語氣禮貌但不太正式，適合職場或團隊溝通

【版本3｜正式版】適合書信或正式場合 語氣正式但溫和 且順暢!!

每個版本後面請附一句「適用情境說明」。
請依照格式輸出，不要多加說明或評論。
"""

# TODO: 使用 LLM 產生輸出
trans_output = call_llm(user_trans_prompt, system=system_trans)


save_text("outputs/task3_translation_tone.txt", trans_output)
print("任務三完成（暫存）。輸出檔：outputs/task3_translation_tone.txt")

任務三完成（暫存）。輸出檔：outputs/task3_translation_tone.txt



---
## 提交與輸出
請在下方設定你的 **學號** 與 **姓名**，系統會將三個任務的輸出合併成一個檔案，並以 `學號_prompt_hw1.txt` 命名，方便上傳。

> 若你實作了實際的 API 呼叫，請同時上傳 `.ipynb` 與最終 `.txt`，並確保不要把 API Key 放進檔案。


In [ ]:

# ===== TODO(學生)：填入你的資訊 =====
student_id = "411286029"   # 例：學號
student_name = "林羽航"

bundle_name = f"{student_id}_prompt_hw1.txt"
out_paths = [
    "outputs/task1_email_formal.txt",
    "outputs/task2_summary_cot.txt",
    "outputs/task3_translation_tone.txt",
]

merged = []
for p in out_paths:
    if os.path.exists(p):
        merged.append(f"===== {os.path.basename(p)} =====\n" + load_text(p) + "\n")
    else:
        merged.append(f"===== {os.path.basename(p)} =====\n(尚未產生)\n")

save_text(os.path.join("outputs", bundle_name), "\n".join(merged))
print(f"✅ 已整合輸出：outputs/{bundle_name}")


✅ 已整合輸出：outputs/411286029_prompt_hw1.txt



---
## 誠信與說明
- 請在 Notebook 中保留你的 Prompt 設計思路與步驟（尤其是任務二的 CoT）。
- 若使用了特定外部資源或工具，請於最終輸出結尾簡述。
- 請勿外流你的 API Key。建議以環境變數或本機設定檔管理。

祝順利完成作業！
